In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os
from dotenv import load_dotenv
load_dotenv()

DATA_LOC = os.getenv("DATA_LOC")
print(DATA_LOC)

~/Data/DCIA_DATA/Data/


### RMBW Data

What is RMBW?
- Regionale Monitor Brede Welvaart
- Maps the braod prosperity of municipalities, provinces and COROP regions using 42 indicators
- broad prosperity concenrs the quality of life here and now and the extent to which this comes at the expense of that of future generations or of people elsewhere in the world

In [6]:
rmbw = pd.read_csv(DATA_LOC + "GENERAL SOURCES/rmbw2025_resultaten.csv")

print(rmbw.columns)
rmbw.head(50)

Index(['ind_code', 'statcode', 'trend_periode', 'trend_kleur', 'positie',
       'pos_van_n', 'pos_jaar', 'pos_kleur', 'een_na_laatste_jaar',
       'laatste_jaar', 'mut_tekst_nl', 'mut_kleur'],
      dtype='str')


,ind_code,statcode,trend_periode,trend_kleur,positie,pos_van_n,pos_jaar,pos_kleur,een_na_laatste_jaar,laatste_jaar,mut_tekst_nl,mut_kleur
0,01.07_NW,NL01,2017-2024,groen,1,1,2024,grijs,2023.0,2024.0,"-6,6%",rood
1,01.07_NW,PV22,2017-2024,groen,5,12,2024,grijs,2023.0,2024.0,"-3,8%",rood
2,01.07_NW,PV24,2017-2024,groen,6,12,2024,grijs,2023.0,2024.0,"-4,2%",rood
3,01.07_NW,PV21,2017-2024,groen,8,12,2024,grijs,2023.0,2024.0,"-4,4%",rood
4,01.07_NW,PV25,2017-2024,groen,3,12,2024,groen,2023.0,2024.0,"-4,1%",rood
5,01.07_NW,PV20,2017-2024,groen,12,12,2024,rood,2023.0,2024.0,"-9,4%",rood
6,01.07_NW,PV31,2017-2024,groen,9,12,2024,grijs,2023.0,2024.0,"-10,1%",rood
7,01.07_NW,PV30,2017-2024,groen,1,12,2024,groen,2023.0,2024.0,"-4,8%",rood
8,01.07_NW,PV27,2017-2024,groen,10,12,2024,rood,2023.0,2024.0,"-12,1%",rood
9,01.07_NW,PV23,2017-2024,groen,7,12,2024,grijs,2023.0,2024.0,"-2,4%",rood


In [9]:
print(rmbw.shape)

(12343, 12)


Columns
- ind_code: Looks like an indicator code
- statcode: city code or province code. From the translation city code would make sense, but looking at examples it seems to be region
- trend_periode: trend period
- 'trend_kleur: trend color
- 'positie': position
- 'pos_van_n': position out of n
- 'pos_jaar': position year
- 'pos_kleur: position colour
- 'een_na_laatste_jaar: one before last year (used to compute the change)
- 'laatste_jaar': last year
- 'mut_tekst_nl': mutation? text, looks like percentage change
- 'mut_kleur': mutation? color



In [7]:
print(rmbw["statcode"].unique())

<StringArray>
[  'NL01',   'PV22',   'PV24',   'PV21',   'PV25',   'PV20',   'PV31',
   'PV30',   'PV27',   'PV23',
 ...
 'GM0355', 'GM0299', 'GM0637', 'GM0638', 'GM1892', 'GM0879', 'GM0301',
 'GM1896', 'GM0642', 'GM0193']
Length: 395, dtype: str


Stat code is either province or city.  We should be able to get province codes and municipality codes from here: https://www.cbs.nl/nl-nl/onze-diensten/methoden/classificaties/overig/gemeentelijke-indelingen-per-jaar/indeling-per-jaar/gemeentelijke-indeling-op-1-januari-2026
- NL01 is probably the whole of the country
- PV is province
- GM is gementee, so municipality

In [8]:
print(rmbw["ind_code"].unique())

<StringArray>
[   '01.07_NW',    '01.10_NW',    '01_GZ_08',    '01_GZ_20',  '01_KL_H_01',
  '01_KL_H_36', '01_KL_OK_08', '01_KL_OK_10', '01_KL_OK_11', '01_KL_OK_14',
 '01_KL_OK_30', '01_KL_OK_39', '01_KL_PK_01', '01_KL_PK_03', '01_KL_PK_07',
 '02_HB_MK_30', '02_HB_NK_30', '02_HB_NK_31', '02_HB_SK_01',   '03_PVI_04',
    '04_VE_03',   '05_OPL_08',   '07_K&E_03',   '09_E&V_03',   '09_E&V_12',
   '09_E&V_13',    '10.31_NW',    '15.15_NW',   '18_FIN_32',     'R_GZ_03',
  'R_HN_AV_02', 'R_HN_MIL_01', 'R_HN_MIL_02',  'R_HN_SL_02',  'R_HN_SL_03',
  'R_HN_SL_04',   'R_HN_V_01',   'R_L_NK_01',   'R_L_NK_02',   'R_L_NK_03',
   'R_L_NK_04',   'R_L_SK_01']
Length: 42, dtype: str


IND code stands for the indicator code. We have 42 separate indicators.

- description of the indicators: https://www.cbs.nl/nl-nl/visualisaties/regionale-monitor-brede-welvaart/beschrijving-indicatoren
- These 42 indicators seem crucial and very useful, from distance to green areas to OV, to cofees, green house gases, GDP. We should be able to build a ton of indicators around this data.

In [17]:
print(rmbw["trend_periode"].unique())

#Only have one period, 2017-2024

<StringArray>
['2017-2024']
Length: 1, dtype: str


This has only position from N no actual indicator values. What is the position compared with, the region?

In [38]:
rmbw["percentile"] = rmbw["positie"] / rmbw["pos_van_n"]

rmbw.head()

,ind_code,statcode,trend_periode,trend_kleur,positie,pos_van_n,pos_jaar,pos_kleur,een_na_laatste_jaar,laatste_jaar,mut_tekst_nl,mut_kleur,percentile
0,01.07_NW,NL01,2017-2024,groen,1,1,2024,grijs,2023.0,2024.0,"-6,6%",rood,1.000000
1,01.07_NW,PV22,2017-2024,groen,5,12,2024,grijs,2023.0,2024.0,"-3,8%",rood,0.416667
2,01.07_NW,PV24,2017-2024,groen,6,12,2024,grijs,2023.0,2024.0,"-4,2%",rood,0.500000
3,01.07_NW,PV21,2017-2024,groen,8,12,2024,grijs,2023.0,2024.0,"-4,4%",rood,0.666667
4,01.07_NW,PV25,2017-2024,groen,3,12,2024,groen,2023.0,2024.0,"-4,1%",rood,0.250000


Pivot table of municipality/province in the top percentile for each category

In [39]:
rmbw_pivot = rmbw.pivot_table(
    index="statcode",
    columns=["ind_code"],
    values="percentile",
    aggfunc="first" #This will only give one value so no need for calculating mean or any other agg func, I checked
)

rmbw_pivot.head(50)

ind_code,01.07_NW,01.10_NW,01_GZ_08,01_GZ_20,01_KL_H_01,01_KL_H_36,01_KL_OK_08,01_KL_OK_10,01_KL_OK_11,01_KL_OK_14,...,R_HN_MIL_02,R_HN_SL_02,R_HN_SL_03,R_HN_SL_04,R_HN_V_01,R_L_NK_01,R_L_NK_02,R_L_NK_03,R_L_NK_04,R_L_SK_01
statcode,,,,,,,,,,,,,,,,,,,,,
CR01,0.800000,0.875000,1.000000,NaN,0.325000,0.975000,0.625000,0.900000,1.000000,0.900000,...,0.375000,0.325000,0.850000,0.850000,NaN,NaN,NaN,0.225000,0.200000,NaN
CR02,0.825000,0.850000,0.950000,NaN,0.100000,0.950000,0.075000,0.750000,0.975000,1.000000,...,0.825000,0.325000,1.000000,0.650000,NaN,NaN,NaN,0.200000,0.225000,NaN
CR03,0.975000,0.350000,0.100000,NaN,0.950000,1.000000,0.875000,0.450000,0.525000,0.875000,...,0.550000,0.650000,0.650000,0.475000,NaN,NaN,NaN,0.325000,0.625000,NaN
CR04,0.775000,0.575000,0.575000,NaN,0.375000,0.900000,0.650000,0.150000,0.450000,0.350000,...,0.125000,0.125000,0.350000,0.625000,NaN,NaN,NaN,0.100000,0.450000,NaN
CR05,0.375000,0.100000,0.450000,NaN,0.075000,0.725000,0.225000,0.125000,0.375000,0.950000,...,0.325000,0.325000,0.850000,0.725000,NaN,NaN,NaN,0.025000,0.350000,NaN
CR06,0.625000,0.450000,0.500000,NaN,0.300000,0.800000,0.900000,0.200000,0.700000,0.550000,...,0.150000,0.325000,0.650000,0.725000,NaN,NaN,NaN,0.175000,0.175000,NaN
CR07,0.250000,0.050000,0.650000,NaN,0.350000,0.250000,0.925000,0.325000,0.325000,0.150000,...,0.025000,0.650000,0.850000,0.875000,NaN,NaN,NaN,0.050000,0.075000,NaN
CR08,0.750000,0.775000,0.975000,NaN,0.150000,0.875000,0.675000,0.375000,0.950000,0.575000,...,0.100000,0.950000,0.850000,0.975000,NaN,NaN,NaN,0.250000,0.050000,NaN
CR09,0.500000,0.175000,0.725000,NaN,0.250000,0.625000,0.800000,0.025000,0.725000,0.175000,...,0.050000,0.650000,0.850000,1.000000,NaN,NaN,NaN,0.125000,0.250000,NaN
